# Explore a market model

In [ ]:
import polars as pl

import numpy as np
import plotly.express as px

from collections import defaultdict

from demand_model import calculate_expected_demand

In [ ]:
n_points = 100000
attractivness = 10
prices = [0, 1, 2, 3, 4]

prices = np.random.rand(n_points) * 3 + 2
base_demand = 5
elasticity = -2
avg_demand = calculate_expected_demand(base_demand, prices, elasticity)


demand_example_data = pl.DataFrame(
    {
        "prices": prices,
        "avg_demand":avg_demand,
        "demand": np.random.poisson(avg_demand),
    }
)

avg_demand = (
    demand_example_data.group_by(pl.col("prices").round(1))
    .agg(pl.mean("demand"), pl.mean("avg_demand"))
    .sort("prices")
)
px.line(avg_demand, x="prices", y=["demand", "avg_demand"]).show()


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


In [ ]:
def calculate_expected_demand(base_demand, price, elasticity):
    """Helper function to calculate expected demand given a price and elasticity and base demand."""
    return base_demand * np.exp(price * elasticity)


In [ ]:
class SingleMarket(gym.Env):
    """One article market environment.

    Demand is given as a Poisson distribution with a rate of the attractiveness minus price.
    """

    def __init__(
        self, starting_stock=10, n_periods=3, price_range=(10, 50), elasticity=-2, attractiveness_mean=500, attractiveness_std=10
    ):
        self.attractiveness_mean = attractiveness_mean
        self.attractiveness_std = attractiveness_std
        self.elasticity = elasticity
        self.starting_stock = starting_stock
        self.n_periods = n_periods
        self.price_range = price_range

        self.action_space = spaces.Discrete(*price_range)

        self.reset()

    def step(self, action):
        price = action
        demand = np.random.poisson(np.max([self.attractiveness - price, 0]))
        sales = np.minimum(demand, self.stock)
        profit = demand * sales

        if self.t > self.n_periods:
            terminate = True
        else:
            terminate = False

        self.t += 1
        self.stock -= sales

        return (
            (self.t, self.stock),
            profit,
            terminate,
            {"sales": sales},
        )

    def reset(self):

        self.base_demand = (
            np.random.randn() * self.attractiveness_std
            + self.attractiveness_mean
        )

        self.stock = self.starting_stock
        self.t = 0
        return (self.t, self.stock)


In [ ]:
market = SingleMarket()
state = market.reset()
print(state)

## Train a model

In [ ]:
class LearningSingleSeller:
    def __init__(self, market):
        self.value_function = defaultdict(lambda: 0)
        self.epsilon = 0.1
        self.alpha = 0.10
        self.learning_method = "sarsa"
        self.learning_rate = "decaying-epsilon"
        self.t = 0
        self.price_range = market.price_range
        self.n_articles = market.n_articles

    def _get_epsilon(self):
        if self.learning_rate == "decaying-epsilon":
            return 1 / (self.t + 1)
        elif self.learning_rate == "constant":
            return 0.1

    def get_action(self, state):
        epsilon = self._get_epsilon()
        if np.random.rand() < epsilon:
            action = np.random.choice(range(*self.price_range), size=self.n_articles)
        else:
            action = np.argmax(
                [self.value_function[state, i] for i in range(env.action_space.n)]
            )
        return int(action)

    def update(self, state, action, reward, next_state, next_action, info):
        if self.learning_method == "sarsa":
            self.value_function[state, action] += self.alpha * (
                reward
                + self.value_function[next_state, next_action]
                - self.value_function[state, action]
            )
        self.t += 1


In [ ]:
market = Market()
agent = LearningSingleSeller(market)
state = test_market.reset()

print(state)

In [ ]:
agent.get_action(state)

In [ ]:
state


## Old Code

In [ ]:
import gymnasium as gym


from collections import defaultdict

import matplotlib.pyplot as plt
import plotly.graph_objects as go

from market import SingleMarket, MultiMarket

In [ ]:

class RandomSingleSeller:
    def __init__(
        self,
    ):
        pass

    def get_action(self, env, state):
        action = env.action_space.sample()
        return action

    def update(self, state, action, reward, next_state, next_action, info):
        pass


In [ ]:
class FixedSingleSeller:
    def __init__(
        self
    ):
        pass

    def get_action(self, env, state):
        return env.action_space.n//2

    def update(self, state, action, reward, next_state, next_action, info):
        pass


In [ ]:
class LearningSingleSeller:
    def __init__(
        self,
    ):
        self.value_function = defaultdict(lambda: 0)
        self.epsilon = 0.1
        self.alpha = 0.10
        self.learning_method = "sarsa"
        self.learning_rate = "decaying-epsilon"
        self.t = 0

    def _get_epsilon(self):
        if self.learning_rate == "decaying-epsilon":
            return 1 / (self.t + 1)
        elif self.learning_rate == "constant":
            return 0.1

    def get_action(self, env, state):
        epsilon = self._get_epsilon()
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(
                [self.value_function[state, i] for i in range(env.action_space.n)]
            )
        return int(action)

    def update(self, state, action, reward, next_state, next_action, info):
        if self.learning_method == "sarsa":
            self.value_function[state, action] += self.alpha * (
                reward
                + self.value_function[next_state, next_action]
                - self.value_function[state, action]
            )
        self.t += 1


In [ ]:
def generate_episodes(env, agent, n_episodes=2):
    sequence = []
    for r in range(n_episodes):
        terminated = False

        # initialize:
        state = env.reset()
        action = agent.get_action(env, state)

        while not terminated:
            # update loop
            next_state, reward, terminated, info = env.step(action)
            next_action = agent.get_action(env, next_state)
            agent.update(state, action, reward, next_state, next_action, info)
            action, state = next_action, next_state

            sequence.append((state, action, reward, r, info))

    episodes = pl.DataFrame(
        sequence,
        schema=["state", "action", "reward", "episode", "info"],
        orient="row",
    )

    episodes = episodes.with_columns(
        pl.col("state").list.get(0).alias("t"),
        pl.col("state").list.get(1).alias("stock"),
    )

    return episodes


In [ ]:
env = SingleMarket(max_price=10, n_periods=10)
agent = RandomSingleSeller()

episodes = generate_episodes(env, agent, n_episodes=2)
episodes

In [ ]:
env = SingleMarket(max_price=10, n_periods=10)
agent = FixedSingleSeller()

episodes = generate_episodes(env, agent, n_episodes=1000)
episodes

px.line(
    episodes.group_by("episode")
    .agg(pl.sum("reward").alias("total_reward"))
    .sort("episode"),
    x="episode",
    y="total_reward",
)

In [ ]:
env = SingleMarket(max_price=4, n_periods=10)
agent = LearningSingleSeller()

episodes = generate_episodes(
    env,
    agent,
    n_episodes=1000,
)
episodes


In [ ]:
value_function = dict(agent.value_function)

In [ ]:
list(value_function.keys())[0]

In [ ]:
px.line(
    episodes.group_by("episode").agg(pl.sum("reward").alias("total_reward")).sort("episode"),
    x="episode", y="total_reward",
)